# Step 1: APTOS 2019 Retinal Fundus Preprocessing & Stratified Split (Google Colab Ready)

1. Connects Google Drive & clones your GitLab repo automatically
2. Downloads **Kaggle APTOS 2019 Blindness Detection** (`train.csv` + `train_images/`)
3. Generates stratified `data/metadata.csv` (80% Train / 10% Val / 10% Test) & backs it up to Google Drive
4. Previews **Ben Graham Retinal Illumination Normalization** across all 5 ICDR grades.


### 1. Google Colab Setup & Auto-Clone Repository


In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://gitlab.com/Shariar-Ahamed/hybrid-retinal-dr-detection-xai.git'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/RetinaRAG_DR_Project')
    (DRIVE_PROJECT_DIR / 'models').mkdir(parents=True, exist_ok=True)
    (DRIVE_PROJECT_DIR / 'data').mkdir(parents=True, exist_ok=True)

    ROOT_DIR = Path('/content/hybrid-retinal-dr-detection-xai')
    if not ROOT_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(ROOT_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(ROOT_DIR), 'pull'], check=True)
    os.chdir(ROOT_DIR)
    print('[Colab] Repo ready at:', ROOT_DIR)
    print('[Colab] Google Drive backup folder:', DRIVE_PROJECT_DIR)
else:
    ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DRIVE_PROJECT_DIR = None
    print('[Local] Project Root:', ROOT_DIR)

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


### 2. Download APTOS 2019 from Kaggle


In [ ]:
# Download & unzip Kaggle APTOS 2019 directly to fast Colab SSD (/content/aptos2019)
import shutil

APTOS_DIR = Path('/content/aptos2019') if IN_COLAB else (ROOT_DIR / 'data' / 'aptos2019')

if IN_COLAB and not (APTOS_DIR / 'train.csv').exists():
    kaggle_dir = Path('/root/.kaggle')
    kaggle_dir.mkdir(exist_ok=True)
    drive_kaggle = Path('/content/drive/MyDrive/kaggle.json')
    if drive_kaggle.exists():
        shutil.copy2(drive_kaggle, kaggle_dir / 'kaggle.json')
        print('[+] Loaded kaggle.json from Google Drive!')
    elif not (kaggle_dir / 'kaggle.json').exists():
        from google.colab import files
        print('[i] Please upload your kaggle.json API token:')
        uploaded = files.upload()
        for fname in uploaded.keys():
            shutil.move(fname, kaggle_dir / 'kaggle.json')
    os.chmod(kaggle_dir / 'kaggle.json', 0o600)

    APTOS_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(['pip', 'install', '-q', 'kaggle'], check=True)
    subprocess.run(['kaggle', 'competitions', 'download', '-c', 'aptos2019-blindness-detection', '-p', str(APTOS_DIR)], check=True)
    subprocess.run(['unzip', '-q', '-o', str(APTOS_DIR / 'aptos2019-blindness-detection.zip'), '-d', str(APTOS_DIR)], check=True)
    print('[+] APTOS 2019 extracted to:', APTOS_DIR)
else:
    print('[+] APTOS 2019 directory ready at:', APTOS_DIR)


### 3. Generate Stratified `data/metadata.csv` & Backup to Google Drive


In [ ]:
from src.preprocess import generate_aptos2019_metadata, apply_ben_graham_preprocessing, DR_CLASS_NAMES

generate_aptos2019_metadata(aptos_dir=str(APTOS_DIR), val_size=0.10, test_size=0.10, seed=42)

if IN_COLAB and DRIVE_PROJECT_DIR is not None:
    shutil.copy2(ROOT_DIR / 'data' / 'metadata.csv', DRIVE_PROJECT_DIR / 'data' / 'metadata.csv')
    print('[+] Backed up metadata.csv to Google Drive!')


### 4. Visual Verification: Raw Fundus vs. Ben Graham Normalization (Grades 0–4)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

df = pd.read_csv(ROOT_DIR / 'data' / 'metadata.csv')
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for grade in range(5):
    sample = df[df['label'] == grade].iloc[0]
    raw = Image.open(APTOS_DIR / sample['image_path']).convert('RGB')
    bg = apply_ben_graham_preprocessing(raw, img_size=224)
    axes[0, grade].imshow(raw.resize((224, 224)))
    axes[0, grade].set_title(f'Raw | Grade {grade}: {DR_CLASS_NAMES[grade]}', fontsize=9)
    axes[0, grade].axis('off')
    axes[1, grade].imshow(bg)
    axes[1, grade].set_title('Ben Graham Normalized', fontsize=9)
    axes[1, grade].axis('off')
plt.tight_layout()
plt.show()
